In [21]:
# crawl_test => 독립적인 모듈
# vscode, cursor, pycharm, eclipse
# import crawl_test
# 실행여부와 상관없이, 바로 모듈 실행

import re, time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://books.toscrape.com/"

# Refactoring => 1개의 긴 문장 코드를 기능별 여러개의 코드로 나누어서 작성.관리하는 프로그래밍 기법
# 1개의 긴 문장 코드 => 너무 길다보니 버그 발생 => 어디에서 어떤 버그
# 코드 1개의 실행을 위해서 사용해야하는 리스크 높음 => 코드를 분산 => 메모리 스레드 => 멀티스레드 & 싱글스레드 => 운영체제

def parse_book(card, category) :
    title = card.h3.a["title"].strip()
    product_url = urljoin(BASE, card.h3.a["href"])
    price_text = card.select_one(".price_color").get_text(strip=True) # strip() =><a> "\n te st \t" </a>
    price_num = re.sub(r'[^0-9\.]', "", price_text)
    price = float(price_num) if price_num else None
    stock_text = card.select_one(".availability").get_text(strip=True)
    stock = "In stock" in stock_text
    rating_word = card.select_one(".star-rating")["class"][1]
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    rating = rating_map.get(rating_word)

    return {
        "title": title,
        "price": price,
        "stock": stock,
        "rating": rating,
        "category": category,
        "product_url": product_url
    }

def crawl_category(cat_url, max_pages = 2) :
    # 미스터리 > 11개 책 정보 크롤링 > 리스트 자료구조 > 리스트 [ ]
    data = []
    next_url = cat_url
    for _ in range(max_pages) :
        html = requests.get(next_url, timeout=10).text
        soup = BeautifulSoup(html, "html.parser")
        category = soup.select_one(".page-header.action h1") # <h1>Sequential Art</h1>
        category = category.get_text(strip=True) if category else "Unknown" # 3항 조건 연산자
        for card in soup.select("ol.row > li.col-xs-6.col-sm-4.col-md-3.col-lg-3") :
            data.append(parse_book(card, category))
        nxt = soup.select_one("li.next a")
        if not nxt :
            break
        next_url = urljoin(next_url, nxt["href"])
        time.sleep(0.5)
    return data

def crawl_few_categories() :
    # test01 = requests.get(BASE)
    # test02 = requests.get(BASE, timeout=10).text
    html = requests.get(BASE, timeout=10).text
    soup = BeautifulSoup(html, "html.parser")
    cat_links = [urljoin(BASE, a["href"]) for a in soup.select(".side_categories > ul > li > ul > li > a")[:3]]

    all_rows = []
    for link in cat_links :
        all_rows.extend(crawl_category(link))
    return all_rows

if __name__ == "__main__" :
    rows = crawl_few_categories()
    print(f"Collected : {len(rows)} rows")
    print(rows)

Collected : 69 rows
[{'title': "It's Only the Himalayas", 'price': 45.17, 'stock': True, 'rating': 2, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/its-only-the-himalayas_981/index.html'}, {'title': 'Full Moon over Noahâ\x80\x99s Ark: An Odyssey to Mount Ararat and Beyond', 'price': 49.43, 'stock': True, 'rating': 4, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/full-moon-over-noahs-ark-an-odyssey-to-mount-ararat-and-beyond_811/index.html'}, {'title': 'See America: A Celebration of Our National Parks & Treasured Sites', 'price': 48.87, 'stock': True, 'rating': 3, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/see-america-a-celebration-of-our-national-parks-treasured-sites_732/index.html'}, {'title': 'Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel', 'price': 36.94, 'stock': True, 'rating': 2, 'category': 'Travel', 'product_url': 'https://books.toscrape.com/vagabonding-an-uncommon-guide-to-the-art-of-long-term-w

In [17]:
list_test = [1, 2, 3, 4]
print(dir(list_test))

['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', 'append', 'clear', 'copy', 'count', 'extend', 'index', 'insert', 'pop', 'remove', 'reverse', 'sort']
